In [6]:
import numpy as np
import matplotlib.pyplot as plt

**Problem Statement,** **Suppose you are starting a company that grows and sells wild mushrooms.Since not all mushrooms are edible, you'd like to be able to tell whether a given mushroom is edible or poisonous based on it's physical attributes
You have some existing data that you can use for this task.
Can you use the data to help you identify which mushrooms can be sold safely?**

In [7]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

**Calculation of entropy**

In [8]:
def compute_entropy(y):
    entropy = 0
    n = len(y)
    if n==0:
        return 0
    p1 = np.sum(y==1) / n   #postive variable fraction
    if p1==0 or p1 == 1:
        entropy = 0 #ofcourse the entropy will be 0, if the node is either of one class completely
    else:    
        entropy = (-p1 * np.log2(p1)) - ((1-p1) * np.log2(1-p1))
 
    
    return entropy

**Splitting function**

In [9]:
def split_dataset(X, node_indices, feature): #here node_indices represent the index of the trainig examples or the samples
    left_indices = []
    right_indices = []
    for i in node_indices:
        if X[i][feature] == 1:   #if the current training sample has this feature then we append it to left node
            left_indices.append(i) 
        else:
            right_indices.append(i)  #otherwise we append it to right node
    return left_indices, right_indices

**Information gain**

In [18]:
def compute_information_gain(X, y, node_indices, feature):
    X_node, y_node = X[node_indices] , y[node_indices]  #X_node represent the number of training examples based on current sample index and y_node represent the target variable
    left_indices , right_indices = split_dataset(X, node_indices, feature) 
    X_left , X_right = X[left_indices] , X[right_indices]
    y_left , y_right = y[left_indices] , y[right_indices]

    parent_entropy = compute_entropy(y_node)  #this the entropy at the current node
    left_entropy = compute_entropy(y_left)    #entropy of the leftnode
    right_entropy = compute_entropy(y_right)  #entropy of the rightnode
    w_left = len(X_left) / len(X_node)  #fraction of datasets in left node out of total nodes
    w_right = len(X_right) / len(X_node)  #fraction of datasets in left node out of total nodes
    weighted_node = (w_left * left_entropy) + (w_right * right_entropy)
    information_gain = parent_entropy - weighted_node
    return information_gain
    
    

**Best feature to split the node**

In [19]:
def get_best_split(X, y, node_indices):   
    features = X.shape[1]  #number of features
    max_info = 0
    best_feature = -1
    for feature in range(features):
        info_gain = compute_information_gain(X, y, node_indices, feature)
        if info_gain > max_info:
            max_info = info_gain
            best_feature = feature
    return best_feature        
        

**Design of Decision Tree**

In [20]:
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    if current_depth == max_depth:  #edge case
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
    best_feature = get_best_split(X, y, node_indices) 
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)
root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]    
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0) 

 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 0
  -- Left leaf node with indices [0, 1, 4, 7]
  -- Right leaf node with indices [5]
- Depth 1, Right: Split on feature: 1
  -- Left leaf node with indices [8]
  -- Right leaf node with indices [2, 3, 6, 9]
